# Aufgabe 6 (Scala: Längste Rundreisen)

_Verwenden Sie ausschließlich aus der Lehrveranstaltung bekannte Scala-Konstrukte, dabei sind `var` und seiteneffektbehaftete Methoden/Klassen/Objekte strikt untersagt._

Ermitteln Sie für ein gegebenes Netz (Graph) von Verbindungen zwischen Ausflugszielen eine Rundreise, die eine maximale Anzahl von Zielen besucht (längster Zyklus).

**a)** Vervollständigen Sie die Funktion `maxLength(css)`, die aus einer Liste `css` von Zyklen (gegeben als Liste der besuchten Knotennummern) den längsten Zyklus liefert.

In [1]:
def maxLength: List[List[Int]] => List[Int] = {
   case Nil => ??? // undefined -> Exception
   case h::ts => ts.foldRight(h)(
      (e, r) => if (e.length > r.length) e else r )
}

defined function maxLength

In [2]:
// Beispiel und Schnelltest:
assert(maxLength(List(Nil)) == Nil); print("✅")
assert(maxLength(List(List(1))) == List(1)); print("✅")
print("|")
assert(maxLength(List(List(11),List(21))) == List(11) || maxLength(List(List(1),List(11))) == List(21)); print("✅")
assert(maxLength(List(List(11,12),List(21))) == List(11,12)); print("✅")
assert(maxLength(List(List(11),List(21,22))) == List(21,22)); print("✅")
print("|")
assert(maxLength(List(List(11,12,13),List(21,22),List(31))) == List(11,12,13)); print("✅")
assert(maxLength(List(List(11,12),List(21,22,23),List(31))) == List(21,22,23)); print("✅")
assert(maxLength(List(List(11),List(21,22),List(31,32,33))) == List(31,32,33)); print("✅")

✅✅|✅✅✅|✅✅✅

Graphen sind in dieser Aufgabe durch eine Liste dargestellt, die für jeden Knoten die Liste der Nachfolger enthält (Adjazenzliste gs; Knoten werden über den Index ihres Eintrags identifiziert).

**b)** Ergänzen Sie die Funktion `makeLongest(gs, rs)`, die den übergebenen, <ins>nicht-leeren</ins> Pfad `rs` zu einem längsten Zyklus erweitert (gibt es <ins>keinen</ins> Zyklus, soll `Nil` zurückgegeben werden). Dazu soll sie alle Nachbarn `n` des Endknotens (`rs.last`) von `rs` betrachten:
- Ist `n` der erste Knoten des Pfades `rs`, wurde ein Zyklus gefunden.
- Ist `n` ein anderer Knoten des Pfads, kann mit `n` <ins>kein</ins> Zyklus gebildet werden, der den ganzen Pfad enthält.
- Andernfalls wird `rs` um `n` erweitert und die Suche mit dem neuen Pfad fortgesetzt.

Von den so bestimmten längsten Zyklen aller Nachbarn soll der längste ausgewählt werden.

In [3]:
def makeLongest: (List[List[Int]], List[Int]) => List[Int] =
   (gs, rs) => maxLength(gs(rs.last).map(n =>
      if (n == rs.head)
         rs ::: List(n) 
      else if (rs.contains(n))
         Nil
      else makeLongest(gs, rs:::List(n))))

defined function makeLongest

In [4]:
// Beispiel und Schnelltest:
assert(makeLongest(List(List(1),List(2),List(0)),List(0,1,2)) == List(0,1,2,0)); print("✅") // 0->1->2->0 (Fall 1)
assert(makeLongest(List(List(1),List(2),List(1)),List(0,1,2)) == Nil); print("✅") // 0->1<=>2 (Fall 2)
assert(makeLongest(List(List(1),List(2),List(0)),List(0)) == List(0,1,2,0)); print("✅") // 0->1->2->0 (Fall 3)
print("|")
assert(makeLongest(List(List(1),List(0,2),List(0)),List(0)) == List(0,1,2,0)); print("✅") // 0<=>1->2->0
assert(makeLongest(List(List(1),List(2,0),List(0)),List(0)) == List(0,1,2,0)); print("✅") // 0<=>1->2->0
assert(makeLongest(List(List(1),List(0,2),List(0)),List(1)) == List(1,2,0,1)); print("✅") // 0<=>1->2->0
assert(makeLongest(List(List(1),List(2,0),List(0)),List(1)) == List(1,2,0,1)); print("✅") // 0<=>1->2->0
assert(makeLongest(List(List(1),List(0,2),List(0)),List(2)) == List(2,0,1,2)); print("✅") // 0<=>1->2->0
assert(makeLongest(List(List(1),List(2,0),List(0)),List(2)) == List(2,0,1,2)); print("✅") // 0<=>1->2->0
print("|")
assert(makeLongest(List(List(1),List(0,2),List(0,1,3),List(0,1,2)),List(0)) == List(0,1,2,3,0)); print("✅") // all back to all previous
assert(makeLongest(List(List(1),List(0,2),List(0,1,3),List(0,1,2)),List(1)) == List(1,2,3,0,1)); print("✅") // all back to all previous
assert(makeLongest(List(List(1),List(0,2),List(0,1,3),List(0,1,2)),List(2)) == List(2,3,0,1,2)); print("✅") // all back to all previous
assert(makeLongest(List(List(1),List(0,2),List(0,1,3),List(0,1,2)),List(3)) == List(3,0,1,2,3)); print("✅") // all back to all previous
print("|")
{
def makeLongestCheck: (List[List[Int]], List[Int], Int) => Boolean = (g,p,l) => makeLongest(g,p) match {
    case c =>
       c.startsWith(p,0) && // starts with given path
       c.length == l && // has expected length (i.e. is longest)
       c.head == c.last && // is a cycle
       c.drop(1).distinct.length == l-1 && // has distinct nodes only
       c.zip(c.drop(1)).forall({case (n,s) => g(n).contains(s)}) // is a valid path in the graph
}
assert(makeLongestCheck(List(List(1,2,3),List(0,2,3),List(0,1,3),List(0,1,2)),List(0),5)); print("✅") // fully connected
assert(makeLongestCheck(List(List(1,2,3),List(0,2,3),List(0,1,3),List(0,1,2)),List(1),5)); print("✅") // fully connected
assert(makeLongestCheck(List(List(1,2,3),List(0,2,3),List(0,1,3),List(0,1,2)),List(2),5)); print("✅") // fully connected
assert(makeLongestCheck(List(List(1,2,3),List(0,2,3),List(0,1,3),List(0,1,2)),List(3),5)); print("✅") // fully connected
}

✅✅✅|✅✅✅✅✅✅|✅✅✅✅|✅✅✅✅

**c)** Setzen Sie in der Funktion `longest` die Suche nach einem längsten Zyklus um, indem Sie für jeden möglichen Startknoten $0$ $\leq$ `n` $<$ `gs.length` die Funktion `makeLongest` aufrufen und das passende Ergebnis auswählen.

In [7]:
def longest: List[List[Int]] => List[Int] = gs => {
    maxLength(for (x <- List.range(0,gs.length)) yield makeLongest(gs, List(x)))
}

defined function longest

In [6]:
// Beispiel und Schnelltest:
{
def longestCheck: (List[List[Int]], Int) => Boolean = (g,l) => longest(g) match {
    case c =>
       c.length == l && // has expected length (i.e. is longest)
       c.head == c.last && // is a cycle
       c.drop(1).distinct.length == l-1 && // has distinct nodes only
       c.zip(c.drop(1)).forall({case (n,s) => g(n).contains(s)}) // is a valid path in the graph
}
assert(longestCheck(List(List(1),List(2),List(0)),4)); print("✅") // 0->1->2->0 (Fall 1)
assert(longestCheck(List(List(1),List(2),List(1)),3)); print("✅") // 0->1<=>2 (Fall 2)
assert(longestCheck(List(List(1),List(2),List(0)),4)); print("✅") // 0->1->2->0 (Fall 3)
print("|")
assert(longestCheck(List(List(1),List(0,2),List(0)),4)); print("✅") // 0<=>1->2->0
assert(longestCheck(List(List(1),List(2,0),List(0)),4)); print("✅") // 0<=>1->2->0
print("|")
assert(longestCheck(List(List(1),List(0,2),List(0,1,3),List(0,1,2)),5)); print("✅") // all back to all previous
print("|")
assert(longestCheck(List(List(1,2,3),List(0,2,3),List(0,1,3),List(0,1,2)),5)); print("✅") // fully connected
}

✅✅✅|✅✅|✅|✅